# Qualitative samples

Best-FID checkpoint images and export manifest.

In [1]:
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
plt.rcParams["axes.grid"] = False
import numpy as np

from gen_cats.evaluation.report_analysis import (
    REPO_ROOT,
    best_run_sample_path,
    best_runs_table,
    ensure_plots_dir,
    load_fid_scores,
    write_stats_csv,
)

CHECKPOINTS = REPO_ROOT / "checkpoints"
PLOTS = ensure_plots_dir("results")
rows = best_runs_table(load_fid_scores())
assets = [{**row, "image": best_run_sample_path(row, CHECKPOINTS)} for row in rows]
assets

[{'model': 'beta_vae',
  'label': '$\\beta$-VAE',
  'mean_fid': 278.2905466980558,
  'slug': '996f47caca39',
  'hyperparameters': {'latent_dim': 128, 'beta': 1.0},
  'n_runs': 12,
  'image': PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/checkpoints/beta_vae/996f47caca39/additional_samples.png')},
 {'model': 'vqvae',
  'label': 'VQ-VAE-1',
  'mean_fid': 243.51046802288295,
  'slug': '3acd0306b276',
  'hyperparameters': {'num_embeddings': 512,
   'feature_map_size': 8,
   'recon_loss': 'mse'},
  'n_runs': 24,
  'image': PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/checkpoints/vqvae/3acd0306b276/additional_samples.png')},
 {'model': 'pixelcnn',
  'label': 'PixelCNN',
  'mean_fid': 267.631482051431,
  'slug': '099fc9ce2263',
  'hyperparameters': {'latent_dim': 128,
   'beta': 1.0,
   'num_embeddings': 512,
   'feature_map_size': 16,
   'recon_loss': 'mse',
   'n_critic': 5,
   'batch_size': 64,
   'noise_schedule': 'linear',
   'base_channels': 64,
   'ddim_s

In [2]:
if not any(a["model"] == "tiny_ldm" and a.get("image") for a in assets):
    for seed in (42, 0, 3407):
        p = REPO_ROOT / f"results/prior_comparison/seed_{seed}/tiny_ldm_samples.png"
        if p.is_file():
            assets.append(
                {
                    "model": "tiny_ldm",
                    "label": "Tiny LDM (prior exp.)",
                    "mean_fid": float("nan"),
                    "slug": "prior_comparison",
                    "image": p,
                }
            )
            break
if not any(a["model"] == "ddim" and a.get("image") for a in assets):
    legacy = sorted((CHECKPOINTS / "old_ddim").glob("*/samples_best.png"))
    if legacy:
        assets.append(
            {
                "model": "ddim",
                "label": "DDIM (legacy)",
                "mean_fid": float("nan"),
                "slug": legacy[0].parent.name,
                "image": legacy[0],
            }
        )
present = [a for a in assets if a.get("image") and Path(a["image"]).is_file()]
len(present)

7

In [3]:
DISPLAY_ORDER = [
    "wgan_gp",
    "sn_gan",
    "vqvae",
    "pixelcnn",
    "beta_vae",
    "tiny_ldm",
    "ddim",
]

SAMPLE_ROWS = 2
SAMPLES_PER_ROW = 4


def split_sample_grid(path: Path, grid: int = 4) -> list[np.ndarray]:
    img = mpimg.imread(path)
    h, w = img.shape[:2]
    crop = (min(h, w) // grid) * grid
    off_y = (h - crop) // 2
    off_x = (w - crop) // 2
    img = img[off_y : off_y + crop, off_x : off_x + crop]
    tile_h, tile_w = crop // grid, crop // grid
    tiles: list[np.ndarray] = []
    for row in range(grid):
        for col in range(grid):
            tiles.append(
                img[row * tile_h : (row + 1) * tile_h, col * tile_w : (col + 1) * tile_w]
            )
    return tiles


def panel_title(item: dict) -> str:
    if item["mean_fid"] == item["mean_fid"]:
        return f"{item['label']} — FID {item['mean_fid']:.1f}"
    return f"{item['label']} — FID n/a"


def stitch_tiles(path: Path, sample_rows: int, samples_per_row: int, gap: int = 2) -> np.ndarray:
    tiles = split_sample_grid(path)[: sample_rows * samples_per_row]
    if not tiles:
        raise ValueError(f"No tiles extracted from {path}")
    tile_h, tile_w = tiles[0].shape[:2]
    channels = tiles[0].shape[2] if tiles[0].ndim == 3 else 1
    canvas = np.ones(
        (
            sample_rows * tile_h + (sample_rows - 1) * gap,
            samples_per_row * tile_w + (samples_per_row - 1) * gap,
            channels,
        ),
        dtype=tiles[0].dtype,
    )
    for row in range(sample_rows):
        for col in range(samples_per_row):
            tile_i = row * samples_per_row + col
            if tile_i >= len(tiles):
                continue
            y0 = row * (tile_h + gap)
            x0 = col * (tile_w + gap)
            canvas[y0 : y0 + tile_h, x0 : x0 + tile_w] = tiles[tile_i]
    return canvas


def make_overview_figure(
    items: list[dict],
    *,
    width: float = 10.5,
    model_cols: int = 2,
    sample_rows: int = SAMPLE_ROWS,
) -> plt.Figure:
    n_models = len(items)
    n_model_rows = (n_models + model_cols - 1) // model_cols
    panel_width = width / model_cols
    panel_height = panel_width * (sample_rows / SAMPLES_PER_ROW)
    title_space = 0.08
    fig_height = n_model_rows * (panel_height + title_space)

    fig = plt.figure(figsize=(width, fig_height))
    gs = fig.add_gridspec(
        n_model_rows,
        model_cols,
        wspace=0.02,
        hspace=0.06,
    )

    for i, item in enumerate(items):
        row, col = divmod(i, model_cols)
        remainder = n_models % model_cols
        if remainder == 1 and i == n_models - 1:
            ax = fig.add_subplot(gs[row, :])
        else:
            ax = fig.add_subplot(gs[row, col])
        ax.imshow(
            stitch_tiles(Path(item["image"]), sample_rows, SAMPLES_PER_ROW, gap=1),
            aspect="equal",
        )
        ax.set_title(panel_title(item), loc="left", fontsize=9, pad=2)
        ax.axis("off")

    for j in range(n_models, n_model_rows * model_cols):
        row, col = divmod(j, model_cols)
        fig.add_subplot(gs[row, col]).axis("off")

    fig.subplots_adjust(left=0.002, right=0.998, top=0.99, bottom=0.002)
    return fig


def make_compare_figure(items: list[dict], *, width: float = 10.5) -> plt.Figure:
    compare_rows, compare_cols = 4, 4
    fig, axes = plt.subplots(1, len(items), figsize=(width, width * 0.22 * len(items)))
    if len(items) == 1:
        axes = [axes]
    for ax, item in zip(axes, items, strict=True):
        ax.imshow(stitch_tiles(Path(item["image"]), compare_rows, compare_cols, gap=1), aspect="equal")
        ax.set_title(panel_title(item), loc="left", fontsize=10, pad=3)
        ax.axis("off")
    fig.subplots_adjust(wspace=0.06, left=0.01, right=0.99, top=0.92, bottom=0.01)
    return fig


ordered = sorted(
    present,
    key=lambda item: DISPLAY_ORDER.index(item["model"])
    if item["model"] in DISPLAY_ORDER
    else len(DISPLAY_ORDER),
)

fig = make_overview_figure(ordered, model_cols=2)
out = PLOTS / "samples_best_per_family.png"
fig.savefig(out, dpi=200, bbox_inches="tight", pad_inches=0.01)
plt.close(fig)

compare = [a for a in ordered if a["model"] in ("wgan_gp", "beta_vae")]
fig = make_compare_figure(compare)
out_gan = PLOTS / "samples_gan_vs_vae.png"
fig.savefig(out_gan, dpi=200, bbox_inches="tight", pad_inches=0.01)
plt.close(fig)
out

PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/notebooks/plots/results/samples_best_per_family.png')

In [4]:
import pandas as pd

manifest = pd.DataFrame(
    [
        {
            "model": a["model"],
            "label": a["label"],
            "slug": a.get("slug", ""),
            "mean_fid": a.get("mean_fid"),
            "image": str(a["image"]),
        }
        for a in present
    ]
)
write_stats_csv(manifest, "qualitative_assets.csv")
manifest

,model,label,slug,mean_fid,image
0,beta_vae,$\beta$-VAE,996f47caca39,278.290547,/Users/pawelp/Desktop/education/pw/deepl/gen-c...
1,vqvae,VQ-VAE-1,3acd0306b276,243.510468,/Users/pawelp/Desktop/education/pw/deepl/gen-c...
2,pixelcnn,PixelCNN,099fc9ce2263,267.631482,/Users/pawelp/Desktop/education/pw/deepl/gen-c...
3,wgan_gp,WGAN-GP,2b449afde8bc,174.402621,/Users/pawelp/Desktop/education/pw/deepl/gen-c...
4,sn_gan,SN-GAN,737a5766e176,226.131552,/Users/pawelp/Desktop/education/pw/deepl/gen-c...
5,tiny_ldm,Tiny LDM (prior exp.),prior_comparison,NaN,/Users/pawelp/Desktop/education/pw/deepl/gen-c...
6,ddim,DDIM (legacy),1ca4ccebe534,NaN,/Users/pawelp/Desktop/education/pw/deepl/gen-c...
